In [ ]:
def hodgkin_huxley(
    T: float = 400,
    dt: float = 0.01,
    mu: float = 0.0,
    sigma: float = 3.0,
    cte: float = 4.0,
    plot: bool = True,
) -> Tuple[np.ndarray, np.ndarray]:
    
    """ descripción.... 
    
        Parameters
    ----------
    T :
        Total simulation time.

    dt :
        Time step.

    mu, sigma :
        Parameters of noise I1 for the neuron.

    cte:
        Constant input current added to the neuron.
        
    plot :
        If True, display the simulation plot. If False, no plot is shown.

    Returns
    -------
    V:
        Membrane potential of the neuron.

    """

    # ======================
    # PARÁMETROS HH
    # ======================
    C = 1.0
    gNa, gK, gL = 120.0, 36.0, 0.3
    ENa, EK, EL = 50.0, -77.0, -54.4

    # ======================
    # FUNCIONES HH
    # ======================
    def get_rates(V):
        am = 0.1*(V+40)/(1 - np.exp(-(V+40)/10))
        bm = 4*np.exp(-(V+65)/18)
        ah = 0.07*np.exp(-(V+65)/20)
        bh = 1/(1 + np.exp(-(V+35)/10))
        an = 0.01*(V+55)/(1 - np.exp(-(V+55)/10))
        bn = 0.125*np.exp(-(V+65)/80)
        return am, bm, ah, bh, an, bn

    # ======================
    # TIEMPO
    # ======================
    n_steps = int(T / dt)
    t_eval = np.linspace(0, T, n_steps)

    # ======================
    # RUIDO (vectorizado)
    # ======================

    I = cte + mu + sigma * np.random.randn(n_steps)

    # ======================
    # INICIALIZACIÓN
    # ======================
    V, m, h, n = np.zeros(n_steps), np.zeros(n_steps), np.zeros(n_steps), np.zeros(n_steps)

    V[0] = -65.0
    am, bm, ah, bh, an, bn = get_rates(V[0])
    m[0] = am/(am+bm)
    h[0] = ah/(ah+bh)
    n[0] = an/(an+bn)

    # ======================
    # BUCLE DE INTEGRACIÓN
    # Canales iónicos mediante integración exponencialy voltaje con Euler explícito 
    # ======================
    for i in range(n_steps - 1):

        am, bm, ah, bh, an, bn = get_rates(V[i])

        tau_m = 1.0 / (am + bm)  
        tau_h = 1.0 / (ah + bh)
        tau_n = 1.0 / (an + bn)
        
        m_inf = am / (am + bm)  
        h_inf = ah / (ah + bh)
        n_inf = an / (an + bn)

        #Canales ionicos mediante integración exponencial
        m[i+1] = m_inf + (m[i] - m_inf) * np.exp(-dt / tau_m) 
        h[i+1] = h_inf + (h[i] - h_inf) * np.exp(-dt / tau_h)                                                   
        n[i+1] = n_inf + (n[i] - n_inf) * np.exp(-dt / tau_n)

        
        I_ion = gNa*(m[i]**3)*h[i]*(V[i]-ENa) + gK*(n[i]**4)*(V[i]-EK) + gL*(V[i]-EL)           
        V[i+1] = V[i] + (I[i] - I_ion) * dt / C   #Euler para el voltaje (mantener dt pequeño)

    
    # ======================
    # PLOTS
    # ======================
    if plot:
        fig, ax1 = plt.subplots(figsize=(10, 6))

        # Eje principal (Potencial de Membrana)
        color_v = "tab:blue"
        ax1.set_xlabel("Tiempo (ms)")
        ax1.set_ylabel("Potencial V (mV)", color=color_v)
        ax1.plot(t_eval, V, color=color_v, linewidth=1.5, label="V(t)")
        ax1.tick_params(axis="y", labelcolor=color_v)

        # Eje secundario para la Corriente Inyectada
        ax2 = ax1.twinx()
        color_i = "tab:orange"
        ax2.set_ylabel("Corriente I (µA/cm²)", color=color_i)
        ax2.plot(
            t_eval, I, color=color_i, alpha=0.5, linestyle="--", label="I(t)"
        )
        ax2.tick_params(axis="y", labelcolor=color_i)

        plt.title("Dinámica de la Neurona: Potencial y Corriente Inyectada")
        fig.tight_layout()
        plt.show()

    return V